# Assignment 12: Predicting Hotel Booking Cancellations  
## Models: Naïve Bayes, Support Vector Machine (SVM), and Neural Network

**Objectives:**
- Understand how to use classification models (Naïve Bayes, SVM, Neural Networks) to predict hotel cancellations.
- Compare models in terms of accuracy, complexity, and business relevance.
- Interpret and communicate model results from a business perspective.

<a href="https://colab.research.google.com/github/Stan-Pugsley/is_4487_base/blob/main/Assignments/assignment_12_bayes_svm_neural.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Hotel Bookings - Business Context
You work as a data analyst for a hospitality group that manages both **Resort** and **City Hotels**. One major challenge in operations is the unpredictability of **booking cancellations**, which affects staffing, inventory, and revenue planning.

You’ve been asked to use historical booking data to predict whether a future booking will be canceled. Your insights will help management plan more effectively.

Your tasks are to:
1. Build and evaluate three models: Naïve Bayes, SVM, and Neural Network.
2. Compare performance.
3. Recommend which model is best suited for the business needs.

### Key Use Cases
- Understand customer booking behavior
- Explore factors related to cancellations
- Segment guests based on booking characteristics
- Compare city vs. resort hotel performance




## Data Dictionary

This dataset contains booking information for two types of hotels: a **city hotel** and a **resort hotel**. Each record corresponds to a single booking and includes various details about the reservation, customer demographics, booking source, and whether the booking was canceled.

**Source**: [GitHub - TidyTuesday: Hotel Bookings](https://github.com/rfordatascience/tidytuesday/blob/master/data/2020/2020-02-11/readme.md)

| Variable | Type | Description |
|----------|------|-------------|
| `hotel` | character | Hotel type: City or Resort |
| `is_canceled` | integer | 1 = Canceled, 0 = Not Canceled |
| `lead_time` | integer | Days between booking and arrival |
| `arrival_date_year` | integer | Year of arrival |
| `arrival_date_month` | character | Month of arrival |
| `stays_in_weekend_nights` | integer | Nights stayed on weekends |
| `stays_in_week_nights` | integer | Nights stayed on weekdays |
| `adults` | integer | Number of adults |
| `children` | integer | Number of children |
| `babies` | integer | Number of babies |
| `meal` | character | Type of meal booked |
| `country` | character | Country code of origin |
| `market_segment` | character | Booking source (e.g., Direct, Online TA) |
| `distribution_channel` | character | Booking channel used |
| `is_repeated_guest` | integer | 1 = Repeated guest, 0 = New guest |
| `previous_cancellations` | integer | Past booking cancellations |
| `previous_bookings_not_canceled` | integer | Past bookings not canceled |
| `reserved_room_type` | character | Initially reserved room type |
| `assigned_room_type` | character | Room type assigned at check-in |
| `booking_changes` | integer | Number of booking modifications |
| `deposit_type` | character | Deposit type (No Deposit, Non-Refund, etc.) |
| `agent` | character | Agent ID who made the booking |
| `company` | character | Company ID (if booking through company) |
| `days_in_waiting_list` | integer | Days on the waiting list |
| `customer_type` | character | Booking type: Contract, Transient, etc. |
| `adr` | float | Average Daily Rate (price per night) |
| `required_car_parking_spaces` | integer | Requested parking spots |
| `total_of_special_requests` | integer | Number of special requests made |
| `reservation_status` | character | Final status (Canceled, No-Show, Check-Out) |
| `reservation_status_date` | date | Date of the last status update |

This dataset is ideal for classification, segmentation, and trend analysis exercises.


## 1. Load and Prepare the Hotel Booking Dataset

**Business framing:**  
Your hotel client wants to understand which bookings are most at risk of being canceled. But before modeling, your job is to prepare the data to ensure clean and reliable input.

### Do the following:
- Import data from the hotels dataset into a dataframe (in GitHub go to the DataSets folder and look for `hotels.csv`)
- Remove or impute missing values
- Encode categorical variables
- Create your `X` (features) and `y` (target = `is_canceled`)
- Split the data into training and test sets (70/30)

### In Your Response:
1. How many total rows and columns are in the dataset?
2. What types of features (categorical, numerical) are included?
3. What steps did you take to clean or prepare the data?


In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split

# 1. Import data from the hotels dataset
github_csv_url = "https://raw.githubusercontent.com/Stan-Pugsley/is_4487_base/main/DataSets/hotels.csv"
df = pd.read_csv(github_csv_url)

# Print initial rows and shape
print("Initial DataFrame shape:", df.shape)
print("\nInitial DataFrame head:\n", df.head())
print("Initial DataFrame info:", df.info())

# 2. Remove or impute missing values
# Drop columns that are either non-predictive, have too many missing values, or lead to data leakage
# 'company' and 'agent' have many NaNs and are IDs, not typically predictive features without further engineering.
# 'reservation_status', 'reservation_status_date', and 'assigned_room_type' can lead to data leakage
# as they are outcomes or are determined after the booking decision.
columns_to_drop = ['company', 'agent', 'reservation_status', 'reservation_status_date', 'assigned_room_type']
df = df.drop(columns=columns_to_drop)

# Impute 'country' with 'Unknown' as a category
df['country'].fillna('Unknown', inplace=True)

# Impute 'children' with 0 (as missing likely means no children)
df['children'].fillna(0, inplace=True)

# Impute 'adr' (Average Daily Rate) with its median to handle potential outliers and NaNs
# Ensure 'adr' is numeric first
if df['adr'].dtype == 'object':
    df['adr'] = pd.to_numeric(df['adr'], errors='coerce')
df['adr'].fillna(df['adr'].median(), inplace=True)

# Remove rows where total guests (adults + children + babies) is zero, as these are invalid bookings
df = df[df['adults'] + df['children'] + df['babies'] > 0]

print("\nDataFrame shape after dropping columns and rows with 0 guests:", df.shape)
print("Missing values after initial cleaning:\n", df.isnull().sum()[df.isnull().sum() > 0]) # Should be empty

# 3. Encode categorical variables
# Identify categorical columns (object type)
categorical_cols = df.select_dtypes(include=['object']).columns

# Apply one-hot encoding, dropping the first category to avoid multicollinearity
df_encoded = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print("\nDataFrame shape after one-hot encoding:", df_encoded.shape)
print("Number of categorical columns encoded:", len(categorical_cols))

# 4. Create your X (features) and y (target = is_canceled)
X = df_encoded.drop('is_canceled', axis=1)
y = df_encoded['is_canceled']

# Ensure all column names are strings for scikit-learn compatibility (important for some models)
X.columns = X.columns.astype(str)

# 5. Split the data into training and test sets (70/30)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

print("\nShape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

print("\nClass distribution in y_train (0=Not Canceled, 1=Canceled):\n", y_train.value_counts(normalize=True))
print("Class distribution in y_test (0=Not Canceled, 1=Canceled):\n", y_test.value_counts(normalize=True))


Initial DataFrame shape: (9404, 32)

Initial DataFrame head:
           hotel  is_canceled  lead_time  arrival_date_year arrival_date_month  \
0  Resort Hotel            0        342               2015               July   
1  Resort Hotel            0        737               2015               July   
2  Resort Hotel            0          7               2015               July   
3  Resort Hotel            0         13               2015               July   
4  Resort Hotel            0         14               2015               July   

   arrival_date_week_number  arrival_date_day_of_month  \
0                        27                          1   
1                        27                          1   
2                        27                          1   
3                        27                          1   
4                        27                          1   

   stays_in_weekend_nights  stays_in_week_nights  adults  ...  deposit_type  \
0                      

/tmp/ipykernel_14895/268522251.py:22: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['country'].fillna('Unknown', inplace=True)
/tmp/ipykernel_14895/268522251.py:25: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try u

### ✍️ Your Response: 🔧
1. There are 119,390 rows and 32 columns in the initial dataset.

2. The features included in the dataset are categorical including hotel, meal, country, market_segment, distribution_channel, which were changed to one-hot encoding. They also include numerical features including adr (continuous), adults, children, babies, stays_in_weekend_nights. The binary target variable is is_canceled (0=not canceled, 1=canceled).

3. I dropped columns that could be a problem to the dataset including company, agent (a lot of missing values), reservation_status and reservation_status_date (data leakage), and assigned_room_type (determined after booking). I filled missing columns such as country (Unknown), children (0), and adr (converted to numeric and filled with median). I dropped any rows where the guest number was 0. I used one-hot encoding to turn categorical data into numeric for the model and dropped the first column to avoid multicollinearity.

## 2. Build a Naïve Bayes Model

**Business framing:**  
Naïve Bayes is a quick, baseline model often used for early testing or simple classification problems.

### Do the following:
- Train a Naïve Bayes classifier on your training data
- Use it to predict on your test data
- Print a classification report and confusion matrix

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. Where might this model be useful for the hotel (e.g. real-time alerts, operational decisions)?


In [3]:
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the Naïve Bayes classifier
nb_model = GaussianNB()

# Train the model on the training data
nb_model.fit(X_train, y_train)

# Make predictions on the test data
y_pred_nb = nb_model.predict(X_test)

# Print classification report and confusion matrix
print("Naïve Bayes Classification Report:\n", classification_report(y_test, y_pred_nb))
print("Naïve Bayes Confusion Matrix:\n", confusion_matrix(y_test, y_pred_nb))


Naïve Bayes Classification Report:
               precision    recall  f1-score   support

           0       1.00      0.34      0.51      2119
           1       0.33      1.00      0.50       701

    accuracy                           0.50      2820
   macro avg       0.66      0.67      0.50      2820
weighted avg       0.83      0.50      0.50      2820

Naïve Bayes Confusion Matrix:
 [[ 718 1401]
 [   3  698]]


### ✍️ Your Response: 🔧
1. The model does not look like it performs very well as the accuracy is only 0.50, which is like random guessing. The recall with cancellations is good at 1.0, but the precision is not great at 0.33. The recall for non-cancellations is not great at 0.34, and the model incorrectly labels many normal bookings as cancellations. The confusion matrix has 1,401 false positives which is not great, and only 3 false negatives meaning almost no missed cancellations. The model is biased towards predicting cancellations. The best metric to judge performance would likely be recall for cancellations as it does not miss cancellations, but it is also not very precise. This is the best if avoiding misssed cancellations is the most important.

2. This model can still be useful in real-time cancellation risk alerts to flag bookings that are likely to cancel. The hotel could also use the cancellation predictions to slightly overbook rooms to maximize occupancy and revenue. The hotel could also set reminders for predicted cancellations and offer incentives to keep the booking. This information can also help the hotel determine staffing if there are more or less cancellations.

## 3. Build a Support Vector Machine (SVM) Model

**Business framing:**  
SVM can model more complex relationships and is useful when customer behavior patterns aren't linear or obvious.

### Do the following:
- Train an SVM classifier (use `linear` kernel)
- Make predictions and evaluate with classification metrics

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.   

### In Your Response:
1. How well does the model perform?  And what metric is best used to judge the performance?
2. In what business situations could SVM provide better insights than simpler models?


In [4]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the SVM classifier with a linear kernel
# Note: This might take a while to run due to the dataset size.
svm_model = SVC(kernel='linear', random_state=42)

# Train the model on the training data
print("Training SVM model... This may take several minutes.")
svm_model.fit(X_train, y_train)
print("SVM training complete.")

# Make predictions on the test data
y_pred_svm = svm_model.predict(X_test)

# Print classification report and confusion matrix
print("\nSVM Classification Report (Linear Kernel):\n", classification_report(y_test, y_pred_svm))
print("\nSVM Confusion Matrix (Linear Kernel):\n", confusion_matrix(y_test, y_pred_svm))


Training SVM model... This may take several minutes.
SVM training complete.

SVM Classification Report (Linear Kernel):
               precision    recall  f1-score   support

           0       0.89      0.96      0.93      2119
           1       0.86      0.64      0.73       701

    accuracy                           0.88      2820
   macro avg       0.87      0.80      0.83      2820
weighted avg       0.88      0.88      0.88      2820


SVM Confusion Matrix (Linear Kernel):
 [[2044   75]
 [ 251  450]]


### ✍️ Your Response: 🔧
1. The SVM model performs strong at 0.88 accuracy. The recall is excellent at identifying non-cancellations at 0.96, and decent at identifying cancellations at 0.64. The confusion matrix only has 75 false positives, but 251 false negatives. It does not do as good at predicting cancellations, but it is far more reliable and practical. The best metric for predicting cancellations is likely the F1 score because it balances precision and recall.

2. SVM is valuable when the data in the model is complex. It can help identify complex booking patterns with interactions between booking lead time, customer type, and seasonality. Because it is more accurate there will be fewer false alarms and more trustworthy predictions. It can also help with revenue management and pricing strategy by guiding dynamic pricing and overbooking limits.

## 4. Build a Neural Network Model

**Business framing:**  
Neural networks are flexible and powerful, though they are harder to explain. They may work well when subtle patterns exist in the data.

### Do the following:
- Build a MLPClassifier model using the neural_network package from sklearn
- Choose a simple architecture (e.g., 2 hidden layers)
- Evaluate accuracy and performance

**NOTE:** With about 10K rows, this model may run very **slow**.  Be prepared to wait up to 10 minutes.  

### In Your Response:
1. How does this model compare to the others?
2. Would the business be comfortable using a “black box” model like this? Why or why not?


In [5]:
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix

# Initialize the MLPClassifier with a simple architecture (e.g., 2 hidden layers)
# solver='adam' is a good default for large datasets
# alpha is the L2 regularization term parameter
# hidden_layer_sizes=(100, 50) means two hidden layers, first with 100 neurons, second with 50
# max_iter is increased to ensure convergence, though early stopping can be used for optimization
mlp_model = MLPClassifier(hidden_layer_sizes=(100, 50), activation='relu', solver='adam',
                          alpha=0.0001, max_iter=500, random_state=42, verbose=True)

# Train the model on the training data
print("Training Neural Network model... This may take several minutes.")
mlp_model.fit(X_train, y_train)
print("Neural Network training complete.")

# Make predictions on the test data
y_pred_mlp = mlp_model.predict(X_test)

# Print classification report and confusion matrix
print("\nNeural Network Classification Report:\n", classification_report(y_test, y_pred_mlp))
print("\nNeural Network Confusion Matrix:\n", confusion_matrix(y_test, y_pred_mlp))


Training Neural Network model... This may take several minutes.
Iteration 1, loss = 5.67131915
Iteration 2, loss = 1.60269511
Iteration 3, loss = 1.13263665
Iteration 4, loss = 0.90952902
Iteration 5, loss = 0.76091035
Iteration 6, loss = 0.85428935
Iteration 7, loss = 0.70517634
Iteration 8, loss = 0.59116265
Iteration 9, loss = 0.57338963
Iteration 10, loss = 0.81775027
Iteration 11, loss = 0.97603680
Iteration 12, loss = 0.69538142
Iteration 13, loss = 0.37835004
Iteration 14, loss = 0.35566037
Iteration 15, loss = 0.42914942
Iteration 16, loss = 0.96858623
Iteration 17, loss = 0.90170764
Iteration 18, loss = 0.35041491
Iteration 19, loss = 0.52738071
Iteration 20, loss = 0.76145475
Iteration 21, loss = 0.41923060
Iteration 22, loss = 0.31677898
Iteration 23, loss = 0.31903530
Iteration 24, loss = 0.38665425
Iteration 25, loss = 0.32956825
Iteration 26, loss = 0.34193955
Iteration 27, loss = 0.33789443
Iteration 28, loss = 0.30694956
Iteration 29, loss = 0.30553420
Iteration 30, los

### ✍️ Your Response: 🔧
1. The neural network is the best performing model so far at 0.89 accuracy. The model does have slightly more false positives than the SVM model, so there is a trade off with this and precision, but it performs better with the F1 score. It also catches more cancellations than the SVM model and may be better balanced.  

2. Yes, I think this business would be okay with using a black box model like this because of the higher accuracy predictions, which would lead to higher revenue and better planning. It also is better at catching cancellations which is valuable for overbooking strategies, revenue optimization, and customer retention.

## 5. Compare All Three Models

### Do the following:
- Print and compare the accuracy of Naïve Bayes, SVM, and Neural Network models
- Summarize which model performed best

### In Your Response:
1. Which model had the best overall accuracy, training time, interpretability, and ease of use.
2. Would you recommend this model for deployment, and why?


In [7]:
from sklearn.metrics import classification_report

# --- Naïve Bayes Metrics ---
nb_report = classification_report(y_test, y_pred_nb, output_dict=True)
nb_accuracy = nb_report['accuracy']
nb_precision_1 = nb_report['1']['precision']
nb_recall_1 = nb_report['1']['recall']
nb_f1_1 = nb_report['1']['f1-score']

# --- SVM Metrics ---
svm_report = classification_report(y_test, y_pred_svm, output_dict=True)
svm_accuracy = svm_report['accuracy']
svm_precision_1 = svm_report['1']['precision']
svm_recall_1 = svm_report['1']['recall']
svm_f1_1 = svm_report['1']['f1-score']

# --- Neural Network Metrics ---
mlp_report = classification_report(y_test, y_pred_mlp, output_dict=True)
mlp_accuracy = mlp_report['accuracy']
mlp_precision_1 = mlp_report['1']['precision']
mlp_recall_1 = mlp_report['1']['recall']
mlp_f1_1 = mlp_report['1']['f1-score']

print("\n--- Model Performance Comparison ---")
print(f"Naïve Bayes:\n  Accuracy: {nb_accuracy:.2f}\n  Precision (Canceled): {nb_precision_1:.2f}\n  Recall (Canceled): {nb_recall_1:.2f}\n  F1-Score (Canceled): {nb_f1_1:.2f}")
print(f"\nSVM (Linear Kernel):\n  Accuracy: {svm_accuracy:.2f}\n  Precision (Canceled): {svm_precision_1:.2f}\n  Recall (Canceled): {svm_recall_1:.2f}\n  F1-Score (Canceled): {svm_f1_1:.2f}")
print(f"\nNeural Network:\n  Accuracy: {mlp_accuracy:.2f}\n  Precision (Canceled): {mlp_precision_1:.2f}\n  Recall (Canceled): {mlp_recall_1:.2f}\n  F1-Score (Canceled): {mlp_f1_1:.2f}")



--- Model Performance Comparison ---
Naïve Bayes:
  Accuracy: 0.50
  Precision (Canceled): 0.33
  Recall (Canceled): 1.00
  F1-Score (Canceled): 0.50

SVM (Linear Kernel):
  Accuracy: 0.88
  Precision (Canceled): 0.86
  Recall (Canceled): 0.64
  F1-Score (Canceled): 0.73

Neural Network:
  Accuracy: 0.89
  Precision (Canceled): 0.80
  Recall (Canceled): 0.74
  F1-Score (Canceled): 0.77


### ✍️ Your Response: 🔧
1. The Neural network model barely beat the SVM model with 0.89 accuracy. The Naive Bayes was the fastest and easiest to interpret, with the SVM having a long training time. It is also easier to interpret, but was not very accurate.

2. I would recommend the Neural Network model, but this would depend. If I need to interpret and find a solid explanation I may go with the SVM because it is easier to interpret. But, overall I think the Neural Network gives the best information for the purpose of predicting cancellations and managing this in terms of revenue, staffing, and overbooking.

## 6. Final Business Recommendation

### In Your Response:
1. In 100 words or less, write a short recommendation to hotel management based on your analysis.

Possible info to include:
- Which model do you recommend implementing?
- What business problem does it help solve?
- Are there any risks or limitations?
- What additional data might improve the results in the future?
2. How does this relate to your customized learning outcome you created in canvas?


### ✍️ Your Response: 🔧
1. Based on the models that have been created to predict cancellations at the hotel, the model I would recommend to increase revenue, overbooking, and staffing would be the Neural Network. This is the most accurate model we were able to create at 89% accuracy, but it can be difficult to interpret at times. If we need to interpret the information, we have another model that we can use if needed.

2. This assignment directly relates to my customized learning outcomes because it is using a tool to build models for improving a business. I have now been exposed to three more models that can help predict with data that is gathered from a business and will help in the process of making good decisions in future business management.

## Submission Instructions
✅ Checklist:
- All code cells run without error
- All markdown responses are complete
- Submit on Canvas as instructed

In [ ]:
!jupyter nbconvert --to html "assignment_12_bayes_svm_neural.ipynb"